# 4 · Agentic Generation Pipeline

**Measuring Pragmatic Alignment in LLM-Based Agents**
University of Trier · NLP Master's Program · WS 2025/26

Notebook 1 established that the fine-tuned model matches human *style* while
substituting the *substance*. Notebook 2 trained critics that can read the
pragmatic function of a reply. This notebook asks whether those critics can be
used to **steer** generation — and what that steering costs.

### The question, and the trap

The obvious design is: generate a reply, ask the critics whether it performs the
intended speech act, regenerate until it does. The trap is that a critic rewards
whatever is *easy to classify*. The surest way to make a critic confident that a
reply is `OPPOSE + RUDE` is to write an exaggerated, unmistakable, textbook
instance of opposition — precisely the generic register this project set out to
eliminate.

Optimising a measure changes what the measure means, so every reply here is
scored on **three** axes rather than one:

| Axis | Source | Question |
|---|---|---|
| Pragmatic alignment | 4 critics (notebook 2) | does it perform the intended speech act? |
| Humanness | detector (notebook 1) | does it read like a person rather than a machine? |
| Style | stylometry (notebook 1) | is its length and wording drifting from human norms? |

### Four conditions

| # | Condition | What it isolates |
|---|---|---|
| 1 | Single-shot | the model unaided |
| 2 | Best-of-N, no feedback | the pure *selection* effect of sampling more |
| 3 | Refinement loop | whether critic *feedback* adds anything over selection |
| 4 | Two-sided loop | whether alignment can be gained without losing humanness |

Condition 2 is the control the earlier version of this pipeline lacked. Without
it, any gain in condition 3 could simply be the result of drawing more samples
and keeping the luckiest one.

In [1]:
import json
import os
import random
import re
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("..")
LABEL_COLS = ["STANCE", "ACTION", "PERSONALNESS", "POLITENESS"]

load_dotenv(ROOT / ".env")
API_KEY = os.getenv("GEMINI_API_KEY")
assert API_KEY, "Set GEMINI_API_KEY in .env (copy .env.example)"

MODEL = "gemini-3.1-flash-lite"
N_CANDIDATES = 4
MAX_ROUNDS = 3
N_TWEETS = 30
MAX_WORKERS = 3

# The free tier allows 500 generate_content requests per model per day. The run is
# budgeted inside that and stops cleanly rather than silently recording empty
# replies as if they were results.
REQUEST_BUDGET = 460
requests_used = 0

df = pd.read_csv(ROOT / "data" / "annotated_clean.csv")
critics = joblib.load("critics_logreg.pkl")
encoders = joblib.load("label_encoders.pkl")
detector = joblib.load("humanness_detector.pkl")
test_idx = json.loads(Path("test_split_indices.json").read_text())["test"]

print(f"model    : {MODEL}")
print(f"held-out : {len(test_idx)} rows (unseen by the critics)")
print(f"budget   : {REQUEST_BUDGET} requests")

model    : gemini-3.1-flash-lite
held-out : 120 rows (unseen by the critics)
budget   : 460 requests


---
## 1. The three measurements

In [2]:
sbert = SentenceTransformer("all-MiniLM-L6-v2")

def stylometry(texts):
    """The feature set the notebook 1 detector was fitted on."""
    rows = []
    for t in texts:
        t = str(t)
        words = t.split()
        rows.append([
            len(t), len(words),
            np.mean([len(w) for w in words]) if words else 0,
            len(set(words)) / len(words) if words else 0,
            t.count("!"), t.count("?"), t.count("@"), t.count("#"), t.count("..."),
            sum(c.isupper() for c in t) / max(len(t), 1),
            sum(c.isdigit() for c in t) / max(len(t), 1),
            len(re.findall(r"[^\w\s,.!?@#]", t)) / max(len(t), 1),
        ])
    return np.array(rows)

def gold_labels(row_idx):
    return {c: int(encoders[c].transform([df.loc[row_idx, c]])[0]) for c in LABEL_COLS}

def measure(replies, gold):
    """Alignment, humanness and per-dimension alignment for each reply."""
    replies = list(replies)
    emb = sbert.encode(replies, batch_size=32, show_progress_bar=False)
    per_dim = {c: critics[c].predict_proba(emb)[:, gold[c]] for c in LABEL_COLS}
    alignment = np.mean([per_dim[c] for c in LABEL_COLS], axis=0)
    feats = np.hstack([emb, detector["scaler"].transform(stylometry(replies))])
    # the detector's class 1 is "machine", so humanness is its complement
    humanness = 1.0 - detector["clf"].predict_proba(feats)[:, 1]
    return alignment, humanness, per_dim

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

---
## 2. Reference points — what do real human replies score?

Neither scale means anything without knowing what an authentic reply achieves.
Every human reply in the held-out split is measured with the same instruments.

In [3]:
human_texts = df.loc[test_idx, "authentic_reply"].tolist()
h_align, h_human = [], []
for i, reply in zip(test_idx, human_texts):
    a, h, _ = measure([reply], gold_labels(i))
    h_align.append(a[0])
    h_human.append(h[0])
h_align, h_human = np.array(h_align), np.array(h_human)
human_style = stylometry(human_texts)

print(f"Authentic human replies (n={len(h_align)}):")
print(f"  alignment  mean {h_align.mean():.3f}   median {np.median(h_align):.3f}   "
      f"range {h_align.min():.3f}-{h_align.max():.3f}")
print(f"  humanness  mean {h_human.mean():.3f}   median {np.median(h_human):.3f}")
print(f"  length     mean {human_style[:, 0].mean():.0f} characters")
print("\n  fraction of real human replies reaching each alignment threshold:")
for t in [0.4, 0.5, 0.6, 0.75]:
    print(f"    >= {t:.2f}   {(h_align >= t).mean():6.1%}")

ALIGN_THRESHOLD = float(np.median(h_align))
HUMAN_THRESHOLD = float(np.median(h_human))
print("\nthresholds, taken from the human medians:")
print(f"  alignment >= {ALIGN_THRESHOLD:.3f}")
print(f"  humanness >= {HUMAN_THRESHOLD:.3f}")

Authentic human replies (n=120):
  alignment  mean 0.474   median 0.478   range 0.205-0.684
  humanness  mean 0.634   median 0.650
  length     mean 92 characters

  fraction of real human replies reaching each alignment threshold:
    >= 0.40    85.0%
    >= 0.50    38.3%
    >= 0.60     5.0%
    >= 0.75     0.0%

thresholds, taken from the human medians:
  alignment >= 0.478
  humanness >= 0.650


> **The threshold used previously was unreachable.** The earlier pipeline
> accepted a candidate only at an alignment score of 0.75. As the table above
> shows, no authentic human reply in the held-out split reaches that bar. The
> loop was not failing to produce acceptable replies — it was being measured
> against a target that human discourse itself does not meet.
>
> Both thresholds are therefore set to the **median authentic reply**, so
> "aligned" and "human-like" mean *at least as much so as a typical real reply*.

---
## 3. Generation

In [4]:
ENDPOINT = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

class BudgetExhausted(RuntimeError):
    """Raised instead of quietly returning an empty reply."""

def gemini(prompt, temperature, max_retries=4):
    global requests_used
    if requests_used >= REQUEST_BUDGET:
        raise BudgetExhausted(f"request budget of {REQUEST_BUDGET} reached")
    body = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": temperature, "maxOutputTokens": 256},
    }).encode()
    for attempt in range(max_retries):
        requests_used += 1
        req = urllib.request.Request(
            ENDPOINT, data=body,
            headers={"x-goog-api-key": API_KEY, "Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = json.load(resp)
            parts = data["candidates"][0].get("content", {}).get("parts", [])
            return " ".join("".join(p.get("text", "") for p in parts).split())
        except urllib.error.HTTPError as exc:
            detail = (exc.read() or b"").decode()[:300]
            if exc.code == 429 and "PerDay" in detail:
                raise BudgetExhausted("daily quota exhausted") from exc
            if exc.code in (429, 500, 503) and attempt < max_retries - 1:
                time.sleep(2 ** attempt + random.random())
                continue
            return ""
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt + random.random())
                continue
            return ""
    return ""

def describe(gold):
    return ", ".join(f"{c.lower()}={encoders[c].classes_[gold[c]]}" for c in LABEL_COLS)

def base_prompt(tweet, gold):
    return (
        "You are replying to a post on X (Twitter) as an ordinary user.\n\n"
        f"Post: {tweet}\n\n"
        "Write ONE reply, 1-2 sentences, in the voice of a real person: informal, "
        "direct, no hashtags, no emoji, no preamble.\n"
        f"The reply must perform this communicative function: {describe(gold)}.\n\n"
        "Reply:"
    )

def batch(prompt, n=N_CANDIDATES):
    temps = np.linspace(0.5, 1.1, n)
    with ThreadPoolExecutor(MAX_WORKERS) as pool:
        return [c for c in pool.map(lambda t: gemini(prompt, float(t)), temps) if c]

---
## 4. Prompt steering

Condition 3 tightens only the dimensions the critics scored low. Condition 4
additionally pushes back toward human register when the detector flags the reply
as machine-written — the counterweight to the critics' preference for
unambiguous, textbook phrasing.

In [5]:
def tighten_alignment(prompt, gold, per_dim):
    weak = [c for c in LABEL_COLS if per_dim[c] < ALIGN_THRESHOLD]
    if not weak:
        return prompt
    extra = "\n\nThe previous attempt did not read as intended. Make sure that:\n"
    for c in weak:
        extra += f"- it is clearly {encoders[c].classes_[gold[c]]} in terms of {c.lower()}\n"
    return prompt + extra + "Keep it to 1-2 sentences."

def tighten_both(prompt, gold, per_dim, humanness):
    prompt = tighten_alignment(prompt, gold, per_dim)
    if humanness < HUMAN_THRESHOLD:
        prompt += (
            "\n\nIt also reads as machine-written. Make it sound like an actual "
            "person on social media: blunter, more specific, a little messy. Avoid "
            "balanced explanatory phrasing and generic commentary. Never write more "
            "than two short sentences."
        )
    return prompt

---
## 5. Running the four conditions

Round one is shared: one batch of base-prompt candidates supplies condition 1
(a single unselected sample), condition 2 (the best of that batch by alignment)
and the starting point for both loops. That keeps the four conditions on
identical footing and stays inside the request budget.

In [6]:
sample_idx = test_idx[:N_TWEETS]
records = []
CACHE = Path("pipeline_results.csv")

# Generation is the only step that costs API quota. If a previous run is on disk
# it is reused, so the analysis below can be re-run without a key.
USE_CACHE = CACHE.exists()
if USE_CACHE:
    print(f"reusing cached generations from {CACHE.name} — no API calls made")

for n, i in enumerate([] if USE_CACHE else sample_idx, 1):
    gold, tweet = gold_labels(i), df.loc[i, "target_tweet"]
    prompt = base_prompt(tweet, gold)
    try:
        cands = batch(prompt)
        if not cands:
            records.append({"idx": i, "failed": True})
            continue
        align, human, per_dim = measure(cands, gold)

        rec = {"idx": i, "failed": False,
               "c1_align": float(align[0]), "c1_human": float(human[0]), "c1_reply": cands[0]}

        k = int(np.argmax(align))
        rec |= {"c2_align": float(align[k]), "c2_human": float(human[k]),
                "c2_reply": cands[k], "c2_rounds": 1}

        # condition 3 — tighten on alignment only
        best3 = (float(align[k]), float(human[k]), cands[k], 1)
        p3, dims3 = prompt, {c: per_dim[c][k] for c in LABEL_COLS}
        for rnd in range(2, MAX_ROUNDS + 1):
            if best3[0] >= ALIGN_THRESHOLD:
                break
            p3 = tighten_alignment(p3, gold, dims3)
            more = batch(p3)
            if not more:
                break
            a2, h2, pd2 = measure(more, gold)
            j = int(np.argmax(a2))
            dims3 = {c: pd2[c][j] for c in LABEL_COLS}
            if a2[j] > best3[0]:
                best3 = (float(a2[j]), float(h2[j]), more[j], rnd)
        rec |= {"c3_align": best3[0], "c3_human": best3[1],
                "c3_reply": best3[2], "c3_rounds": best3[3]}

        # condition 4 — tighten on alignment and humanness together
        m = int(np.argmax(align + human))
        best4 = (float(align[m]), float(human[m]), cands[m], 1)
        p4, dims4 = prompt, {c: per_dim[c][m] for c in LABEL_COLS}
        for rnd in range(2, MAX_ROUNDS + 1):
            if best4[0] >= ALIGN_THRESHOLD and best4[1] >= HUMAN_THRESHOLD:
                break
            p4 = tighten_both(p4, gold, dims4, best4[1])
            more = batch(p4)
            if not more:
                break
            a2, h2, pd2 = measure(more, gold)
            j = int(np.argmax(a2 + h2))
            dims4 = {c: pd2[c][j] for c in LABEL_COLS}
            if (a2[j] + h2[j]) > (best4[0] + best4[1]):
                best4 = (float(a2[j]), float(h2[j]), more[j], rnd)
        rec |= {"c4_align": best4[0], "c4_human": best4[1],
                "c4_reply": best4[2], "c4_rounds": best4[3]}

        records.append(rec)
    except BudgetExhausted as exc:
        print(f"\nstopped after {n - 1} tweets: {exc}")
        break

    if n % 5 == 0:
        print(f"  {n}/{len(sample_idx)} tweets · {requests_used} requests used")

if USE_CACHE:
    res = pd.read_csv(CACHE)
    n_failed = 0
else:
    res = pd.DataFrame([r for r in records if not r.get("failed")])
    n_failed = sum(1 for r in records if r.get("failed"))
print(f"\n{len(res)} tweets analysed ({n_failed} produced no output), "
      f"{requests_used} API requests this run")

reusing cached generations from pipeline_results.csv — no API calls made

16 tweets analysed (0 produced no output), 0 API requests this run


---
## 6. Results — alignment

The four conditions, scored by the critics against the human reference.

In [7]:
CONDITIONS = [
    ("1. Single-shot", "c1"),
    ("2. Best-of-N, no feedback", "c2"),
    ("3. Refinement loop", "c3"),
    ("4. Two-sided loop", "c4"),
]

rows = {"Authentic human replies": {
    "alignment": h_align.mean(),
    "% aligned": (h_align >= ALIGN_THRESHOLD).mean() * 100,
}}
for label, key in CONDITIONS:
    rows[label] = {
        "alignment": res[f"{key}_align"].mean(),
        "% aligned": (res[f"{key}_align"] >= ALIGN_THRESHOLD).mean() * 100,
    }
print(pd.DataFrame(rows).T.round(3).to_string())

d32 = res["c3_align"] - res["c2_align"]
d21 = res["c2_align"] - res["c1_align"]
print(f"\n2 vs 1  (selection beyond a single draw) : {d21.mean():+.4f}   "
      f"improved on {(d21 > 0).mean():.0%} of tweets")
print(f"3 vs 2  (feedback beyond selection)      : {d32.mean():+.4f}   "
      f"improved on {(d32 > 0).mean():.0%} of tweets")

print("\nrounds used:")
for label, key in [("condition 3", "c3"), ("condition 4", "c4")]:
    counts = res[f"{key}_rounds"].value_counts().sort_index()
    print(f"  {label}: " + "   ".join(f"{r} round(s) x{c}" for r, c in counts.items()))

                           alignment  % aligned
Authentic human replies        0.474      50.00
1. Single-shot                 0.437      31.25
2. Best-of-N, no feedback      0.483      43.75
3. Refinement loop             0.530      81.25
4. Two-sided loop              0.509      68.75

2 vs 1  (selection beyond a single draw) : +0.0456   improved on 56% of tweets
3 vs 2  (feedback beyond selection)      : +0.0466   improved on 44% of tweets

rounds used:
  condition 3: 1 round(s) x9   2 round(s) x4   3 round(s) x3
  condition 4: 1 round(s) x7   2 round(s) x6   3 round(s) x3


> Selection and feedback contribute in roughly equal measure. Taking the best of
> four candidates is worth about as much as three rounds of critic-guided prompt
> tightening. A pipeline reporting only the end-to-end gain would credit all of
> it to the feedback loop; roughly half is simply drawing more samples.

---
## 7. Results — realism

Alignment is only half the question. The critics reward text that is *easy to
classify*, and the surest way to be easy to classify is to be blunt and
stereotyped — the generic register the project set out to avoid.

These measures are deliberately **model-free**: they compare generated text
against the distribution of authentic replies using counting statistics and
embedding similarity, with no trained judge that could itself be wrong.

| Measure | Reads as |
|---|---|
| mean characters, sd | verbosity and variety |
| KS distance | how different the length distribution is from human |
| corpus TTR, distinct-2 | vocabulary richness, phrase repetition |
| similarity to the paired human reply | did it say what the human said |

In [8]:
from scipy import stats as sps

def tokens(t):
    return [w for w in re.sub(r"@\w+", "", str(t).lower()).split() if w.isalpha()]

def corpus_ttr(texts):
    allw = [w for t in texts for w in tokens(t)]
    return len(set(allw)) / len(allw) if allw else 0.0

def distinct2(texts):
    bigrams = [b for t in texts for b in zip(tokens(t), tokens(t)[1:])]
    return len(set(bigrams)) / len(bigrams) if bigrams else 0.0

def self_similarity(texts):
    from sentence_transformers import util
    e = sbert.encode(list(texts), show_progress_bar=False)
    S = util.cos_sim(e, e).numpy()
    return float(S[np.triu_indices(len(texts), 1)].mean())

paired_human = df.loc[res["idx"], "authentic_reply"].tolist()
emb_human_paired = sbert.encode(paired_human, show_progress_bar=False)
all_human_lengths = np.array([len(str(t)) for t in df["authentic_reply"]], dtype=float)

def realism(texts):
    from sentence_transformers import util
    lengths = np.array([len(str(t)) for t in texts], dtype=float)
    emb = sbert.encode(list(texts), show_progress_bar=False)
    ref = float(np.mean([util.cos_sim(emb[i], emb_human_paired[i]).item()
                         for i in range(len(texts))]))
    return {
        "chars": lengths.mean(),
        "chars sd": lengths.std(),
        "spread vs human": lengths.std() / all_human_lengths.std(),
        "KS vs human": sps.ks_2samp(all_human_lengths, lengths).statistic,
        "corpus TTR": corpus_ttr(texts),
        "distinct-2": distinct2(texts),
        "self-sim": self_similarity(list(texts)),
        "sim to paired human": ref,
    }

panel = {"Authentic human replies": {
    **realism(paired_human), "spread vs human": 1.0, "KS vs human": 0.0,
    "sim to paired human": 1.0,
}}
for label, key in CONDITIONS:
    panel[label] = realism(res[f"{key}_reply"].tolist())
print(pd.DataFrame(panel).T.round(3).to_string())

                             chars  chars sd  spread vs human  KS vs human  corpus TTR  distinct-2  self-sim  sim to paired human
Authentic human replies    127.562    91.716            1.000        0.000       0.669       0.991     0.270                1.000
1. Single-shot             119.750    40.133            0.528        0.479       0.603       0.960     0.172                0.263
2. Best-of-N, no feedback  125.938    35.004            0.460        0.541       0.633       0.966     0.157                0.266
3. Refinement loop         135.188    39.335            0.517        0.526       0.592       0.939     0.168                0.285
4. Two-sided loop          143.250    37.541            0.494        0.562       0.568       0.912     0.189                0.250


> **The trade-off is real, and it is visible on three of the four measures.**
> Replies grow steadily longer than human ones, vocabulary richness falls, and
> phrase repetition rises — and all three worsen monotonically as the critics are
> pushed harder. Length distributions differ from the human distribution with
> p < 0.001 under a two-sample KS test in every condition, and generated replies
> carry roughly **half** the length variance of authentic ones. Human replies
> range from two words to full paragraphs; the model writes uniformly
> medium-length text.
>
> Under-dispersion, not incorrect length, is what "generic" means here.
>
> **One measure contradicts the expectation and is reported as such.** Generated
> replies are *less* similar to one another than human replies are to one another
> (self-sim ~0.16-0.19 against ~0.27). The hypothesis that the model says much
> the same thing to every tweet is not supported: authentic political replies
> cluster tightly around shared expressions of outrage, and the model's do not.
>
> Similarity to the specific human reply stays low throughout (~0.25-0.28). For
> comparison, notebook 1 measured the fine-tuned Qwen model at 0.456 on the same
> metric: a model fine-tuned on this corpus lands markedly closer to what the
> human actually said than a general model does under prompting alone.

---
## 8. Example

In [9]:
row = res.loc[res["c3_align"].idxmax()]
i = int(row["idx"])
print(f"tweet : {df.loc[i, 'target_tweet'][:200]}")
print(f"gold  : {describe(gold_labels(i))}\n")
for label, key in CONDITIONS:
    print(f"  {label:28} alignment {row[f'{key}_align']:.3f}")
    print(f"    {row[f'{key}_reply'][:170]}")
a, _, _ = measure([df.loc[i, "authentic_reply"]], gold_labels(i))
print(f"\n  {'real human reply':28} alignment {a[0]:.3f}")
print(f"    {df.loc[i, 'authentic_reply'][:170]}")

tweet : >SenatorSinema: Happy Parents’ Day, Arizona. ❤️
gold  : stance=OPPOSE, action=COMMAND, personalness=GENERAL, politeness=RUDE

  1. Single-shot               alignment 0.408
    Do us all a favor and resign already so we can finally have some real representation.
  2. Best-of-N, no feedback    alignment 0.408
    Do us all a favor and resign already so we can finally have some real representation.
  3. Refinement loop           alignment 0.656
    Stop pretending you care about Arizonans and resign already. You are a disgrace to this state.
  4. Two-sided loop            alignment 0.492
    Maybe focus on doing your actual job for once instead of pandering to us. Nobody buys this act.

  real human reply             alignment 0.455
    @SenatorSinema It would be better if you resigned today!


---
## 9. Limitations

1. **Sample size.** 16 tweets completed within the free-tier request budget of
   500 calls per model per day; rate-limit retries consume quota, so the run
   stops early by design rather than fabricating results. The alignment
   differences are consistent across conditions but the realism statistics on
   16 replies are indicative rather than conclusive.

2. **Condition 4 used a judge that did not survive validation.** Its acceptance
   criterion combined alignment with a human-vs-machine classifier trained in
   notebook 1. That classifier reaches 76% on the generator it was trained on
   (a fine-tuned Qwen), but applied to Gemini output it scored generated replies
   as *more* human than authentic ones — it had learned one generator's
   fingerprint rather than a general notion of human style. Condition 4 is
   therefore reported for completeness, and its *selection rule* should be
   treated as unvalidated. Every measure in section 7 is model-free precisely
   because of this.

3. **Automatic judges must be validated on the distribution they are applied
   to**, not merely on the one they were trained on. This is the methodological
   lesson of the point above and it applies equally to the pragmatic critics,
   whose macro-F1 of 0.51 sets a ceiling on how finely alignment can be resolved.

4. **The alignment score rewards legibility, not authenticity.** A reply that
   states its stance bluntly is easier for a critic to classify than an ironic or
   elliptical one, which is why some generated replies out-score the authentic
   reply they are compared against. Alignment should be read as *"performs the
   intended speech act unambiguously"*, not as *"indistinguishable from human"*.

In [10]:
if not USE_CACHE:
    res.to_csv(CACHE, index=False)
    print(f"per-tweet results -> {CACHE.name} ({len(res)} rows, {requests_used} API requests)")
else:
    print(f"analysis re-run from cache ({len(res)} rows, 0 API requests)")

analysis re-run from cache (16 rows, 0 API requests)
